# 4.4 Streaming & Interrupts

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

**Prerequisites:** 4.2 (state, nodes, edges), 4.3 (the explicit agent graph). The streaming demos run without an API key; the token-streaming section reuses the 4.3 agent and needs an `OPENAI_API_KEY` (or Ollama).

## Learning objectives

- Choose the right stream mode for the job: `values`, `updates`, `messages`, or `custom`, and combine modes without hitting the tuple gotcha.
- Stream LLM tokens with `stream_mode="messages"` and filter the stream by node.
- Emit custom progress events from nodes with `get_stream_writer`.
- Pause a graph mid-run with `interrupt()`, read the pause payload from `__interrupt__`, and resume with `Command(resume=...)` on the same thread.
- Explain why interrupts need a checkpointer, what re-runs on resume, and when to use static `interrupt_before` / `interrupt_after` breakpoints instead.

## Why stream at all

Streaming solves two different problems, and confusing them leads to bad design. The first is latency perception: an agent run takes seconds or minutes, and a UI that shows nothing until the end feels broken. Token streaming fixes the feeling, not the latency. The second is visibility: when a long graph runs, you want to see which node is working, what it returned, and where it stalled. Progress events fix that.

LangGraph's `stream()` serves both, through modes. Pick the mode by asking what the consumer of the stream needs: the full state, the changes, the tokens, or your own events.

## The four modes

| Mode | Yields | Use it for |
|---|---|---|
| `values` | the complete state after every step | debugging, replaying history, a UI that renders the whole ticket |
| `updates` | only the keys each node changed, labelled by node name | a "node X finished" progress feed; cheaper than `values` on large state |
| `messages` | `(message_chunk, metadata)` tuples of LLM tokens | chat UIs, live answer rendering |
| `custom` | whatever your nodes write via `get_stream_writer()` | progress bars, "searching 3 of 5 sources", your own event protocol |

Three more exist for tooling: `checkpoints` (checkpoint metadata), `tasks` (task starts and finishes), and `debug` (both, with extra detail). You will meet them when you profile a slow graph; this lesson covers the four you use daily.

Run this two-node graph once and feel the difference between the first two modes:

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class S(TypedDict):
    n: int
    log: Annotated[list, operator.add]

def a(state: S) -> dict:
    return {"n": state["n"] + 1, "log": ["a"]}

def b(state: S) -> dict:
    return {"n": state["n"] * 10, "log": ["b"]}

builder = StateGraph(S)
builder.add_node("a", a)
builder.add_node("b", b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)
graph = builder.compile()

print("--- values ---")
for chunk in graph.stream({"n": 1, "log": []}, stream_mode="values"):
    print(chunk)

print("--- updates ---")
for chunk in graph.stream({"n": 1, "log": []}, stream_mode="updates"):
    print(chunk)

Expected output:

In [ ]:
--- values ---
{'n': 2, 'log': ['a']}
{'n': 20, 'log': ['a', 'b']}
--- updates ---
{'a': {'n': 2, 'log': ['a']}}
{'b': {'n': 20, 'log': ['b']}}

`values` sends the whole board after each step, which is convenient and wasteful: on a state with a long message history, every step re-sends the entire history. `updates` sends only what changed, scoped by node name, which is the natural shape for a progress feed. Lesson 4.2 used `updates` for the same reason: when you debug a graph, you read what each node returned, not the whole board repeated.

## The tuple gotcha: combining modes

You can pass several modes at once: `stream_mode=["updates", "messages"]`. Do it and the chunk shape changes. With one mode, `stream()` yields the raw chunk. With several, it yields `(mode, chunk)` tuples so you can tell them apart:

In [ ]:
for chunk in graph.stream({"n": 1, "log": []}, stream_mode=["updates", "custom"]):
    print(chunk)

In [ ]:
('custom', {'event': 'a_start', 'n': 1})
('updates', {'a': {'n': 2, 'log': ['a']}})
('custom', {'event': 'b_start', 'n': 2})
('updates', {'b': {'n': 20, 'log': ['b']}})

The most common streaming bug in production code is a consumer written for single-mode chunks that breaks the day someone adds a second mode. If your stream consumer ever does `chunk["messages"]` directly, it will crash on a tuple the moment the mode list grows. Destructure defensively: write the consumer for the tuple form whenever more than one mode is possible.

## Token streaming with `messages` mode

`stream_mode="messages"` yields `(message_chunk, metadata)` tuples, token by token, from every LLM call in the graph. The metadata tells you which node produced the tokens (`metadata["langgraph_node"]`), so a graph with two model nodes can stream each one's tokens to a different place. Run it against the 4.3 agent graph:

In [ ]:
from langchain_core.messages import HumanMessage

for chunk, metadata in agent_graph.stream(
    {"messages": [HumanMessage("What is 12 * 13?")]},
    stream_mode="messages",
):
    if metadata.get("langgraph_node") == "agent" and chunk.content:
        print(chunk.content, end="", flush=True)
print()

Two practical notes. First, the filter by `langgraph_node` matters: without it, tokens from every model call in the graph (a summariser node, a classifier node) land in the same stream, and your chat UI shows internal reasoning it should never display. Second, a model call you do not want streamed can be tagged out: tag the invocation with `"nostream"` and its tokens are skipped in `messages` mode while the call itself still runs. Use it for internal model calls whose output feeds the next node, not the user.

Streaming changes nothing about cost or latency. The same tokens are generated at the same speed; they just reach the UI earlier. If a run is slow, the fix is a smaller model, fewer tool calls, or shorter context, not a stream mode.

## Custom events: progress your UI can use

Tokens tell the user the model is thinking; custom events tell the UI what the graph is doing. A node emits them through the stream writer:

In [ ]:
from langgraph.config import get_stream_writer

def search_sources(state: S) -> dict:
    writer = get_stream_writer()
    sources = ["docs", "changelog", "forum"]
    for i, source in enumerate(sources, start=1):
        writer({"event": "progress", "step": i, "of": len(sources), "source": source})
        # ... actually search the source ...
    return {"log": [f"searched {len(sources)} sources"]}

Custom events are only visible when `"custom"` is in the stream modes: `graph.stream(input, stream_mode="custom")` yields each dict the writer received, nothing else. Without the mode, the writes are silently discarded, which is the second most common streaming bug: the node code is correct, the consumer sees nothing, and the fix is one string in the `stream()` call.

Design your event dicts like a small protocol: a stable `event` key, the fields the UI needs, nothing the UI does not. Frontend code that switches on `event` survives refactors; frontend code that pattern-matches on free-form strings does not.

## Interrupts: pausing for a human

Streaming shows the run; interrupts stop it. Call `interrupt()` anywhere in a node and the graph pauses at that exact point, saves its state, and returns the payload you passed to the caller. Later, you resume by invoking the graph again with `Command(resume=value)` and the same thread id, and the value you passed becomes the return value of `interrupt()` inside the node.

The classic use is approval before an irreversible action: sending a refund, deleting data, emailing a customer. Here is the smallest honest version, with no LLM so you can run it now:

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

class S(TypedDict, total=False):
    draft: str
    status: str

def draft(state: S) -> dict:
    return {"draft": "Docs gap: ef_search"}

def review(state: S) -> dict:
    decision = interrupt({"question": "File this issue?", "draft": state["draft"]})
    return {"status": "filed" if decision else "discarded"}

builder = StateGraph(S)
builder.add_node("draft", draft)
builder.add_node("review", review)
builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
builder.add_edge("review", END)

graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "issue-42"}}

paused = graph.invoke({}, config)
print("paused:", paused["__interrupt__"][0].value)
# paused: {'question': 'File this issue?', 'draft': 'Docs gap: ef_search'}

# ... minutes, hours, or a deploy later, in the same or another process ...
resumed = graph.invoke(Command(resume=True), config)
print("status:", resumed["status"])   # status: filed

Read the run in order and four mechanics become clear:

1. **The payload must be JSON-serializable.** It is written to the checkpoint, and a non-serializable value fails loudly at pause time (`TypeError: Type is not msgpack serializable`). The payload is what you show the reviewer, so keep it small and readable: the question, the action, the amounts. Not the whole state.
2. **`__interrupt__` is how you detect the pause.** The paused result is a state dict with an `__interrupt__` key holding the `Interrupt` objects; `[0].value` is your payload. Check for the key before reading the final state, or you will mistake a paused run for a finished one.
3. **The thread id is the address of the paused run.** Resume with the same `thread_id` and you continue; a new `thread_id` starts a fresh run that knows nothing about the pause. Treat thread ids like primary keys: one per ticket, one per approval, generated once and stored.
4. **On resume, the interrupted node runs again from the top.** Everything before `interrupt()` in that node executes a second time. Keep irreversible side effects after the `interrupt()` call, never before it: prepare the refund before the pause, issue it after the resume. A node that sends the email and then asks for approval has the order backwards.

## The checkpointer is not optional

Interrupts need a checkpointer, and the failure mode is specific enough to state precisely. Without one, the graph still pauses: `invoke` returns the `__interrupt__` payload as if everything were fine. But the paused state was never saved, so resuming fails:

In [ ]:
RuntimeError: Cannot use Command(resume=...) without checkpointer

The pause works and the resume does not, which is worse than a clean error at pause time, because the bug only surfaces when the human answers, possibly hours later. Attach the checkpointer when you compile, every time: `InMemorySaver` for development (state dies with the process), a durable saver for anything real. The SQLite checkpointer survives restarts, which is what an approval that takes hours needs:

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver   # pip install langgraph-checkpoint-sqlite

conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(conn)

`check_same_thread=False` is load-bearing: LangGraph may touch the connection from a different thread than the one that opened it. Lesson 5.1 covers persistence, checkpoint namespaces, and time travel in depth; for this lesson, the rule is one line: no checkpointer, no resume.

## Static breakpoints: `interrupt_before` and `interrupt_after`

`interrupt()` is dynamic: it can sit anywhere in a node and fire conditionally on your logic. Static breakpoints are the compile-time version: pause before or after named nodes, unconditionally.

In [ ]:
graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["review"],
)

The run pauses before `review` executes; `invoke(None, config)` on the same thread continues it. Use static breakpoints for debugging ("pause before the tools node so I can inspect the tool calls") and for uniform policy ("every run pauses before any tool executes"). Use dynamic `interrupt()` when the pause depends on the data: only refunds above 50 EUR need approval, only deletions need confirmation. Lesson 4.3 previewed `interrupt_before=["tools"]` on the agent graph; here is that pattern doing its job:

In [ ]:
agent_graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["tools"],
)

Every tool call now pauses for approval, and the reviewer sees the exact arguments the model chose. Resume with `Command(resume="approved")` or reject with `Command(resume="rejected")`; the tools node reads the decision from the resume value you define. Lesson 5.2 builds the full approval UX on this foundation.

## Streaming a pause

When a streamed run hits an interrupt, the stream yields an `__interrupt__` chunk instead of failing:

In [ ]:
for chunk in graph.stream({}, config, stream_mode="updates"):
    print(chunk)
# {'__interrupt__': (Interrupt(value={'question': 'File this issue?', ...}),)}

A UI consuming the stream should handle that chunk as a first-class event: render the approval card, stop the spinner, wait. Treating the end of the stream as "the run finished" is the bug; the run paused, and the stream ending is the signal to ask the human.

## What can go wrong

- **Resuming on a new thread id.** The checkpointer keys everything by thread. A resume with a fresh id starts an empty run, and the paused state sits orphaned. Generate the thread id once per unit of work and pass it through your whole stack: the API handler, the approval UI, the resume call.
- **Pausing without a checkpointer.** The pause looks fine and the resume raises `RuntimeError`. Attach the checkpointer at compile time; make it a code-review checklist item on any graph that calls `interrupt()`.
- **Side effects before the `interrupt()` call.** The node re-runs on resume, so anything before the pause happens twice. Prepare before, act after. If the preparation itself is expensive, cache it in state so the second pass is cheap.
- **Non-serializable interrupt payloads.** Model objects, file handles, and lambdas fail at pause time with a serialization error. Pass plain data: dicts, strings, numbers. The payload is a message to a human, not a program state dump.
- **Single-mode consumers breaking on combined modes.** `stream()` yields raw chunks for one mode and `(mode, chunk)` tuples for several. Write consumers for the tuple form, or pin the call to exactly one mode and comment why.
- **Custom events with no `"custom"` mode.** `get_stream_writer()` writes are discarded unless the stream request includes the mode. The node is fine; the `stream()` call is missing a string.
- **`messages` mode showing internal calls.** Every model call in the graph streams tokens, including summariser and classifier nodes. Filter by `metadata["langgraph_node"]`, or tag internal calls with `"nostream"`. A chat UI that leaks the summariser's tokens is a design bug, not a framework bug.
- **State containing personal data, now checkpointed.** The moment a checkpointer is attached, every state snapshot is stored, and state often contains user messages. For the EU audience this is a GDPR design point from day one: keep personal data out of state fields you do not need, set a retention policy on the checkpoint store, and know where the snapshots live before the first production run. Lesson 6.5 returns to this with the full compliance picture.

## Check your understanding

**1. What is the difference between `stream_mode="values"` and `stream_mode="updates"`, and when is `updates` the better choice?**
`values` yields the complete state after every step; `updates` yields only the keys each step changed, labelled by node name. `updates` is better when state is large (a long message history) and the consumer only needs to know what each node did, such as a progress feed or a debugger reading per-node returns. `values` re-sends unchanged fields every step, which wastes bandwidth on large state.

**2. You call `graph.stream(input, stream_mode=["updates", "messages"])` and your consumer crashes on `chunk["messages"]`. What changed, and how do you fix the consumer?**
With more than one mode, `stream()` yields `(mode, chunk)` tuples instead of raw chunks, so `chunk` is a tuple and subscripting it fails. Fix the consumer to destructure: `for mode, data in graph.stream(...)`, and branch on `mode`. Alternatively pin the call to a single mode, but the tuple form is the robust default.

**3. Your node writes progress events with `get_stream_writer()`, but the consumer sees nothing. Name two possible causes.**
One: the `stream()` call does not include `"custom"` in its modes, so the writes are discarded. Two: the consumer requested `"custom"` but reads the wrong shape, for example expecting single-mode dicts while the call passes several modes and yields `(mode, data)` tuples.

**4. A graph with `interrupt()` pauses correctly, but resuming raises `RuntimeError: Cannot use Command(resume=...) without checkpointer`. Explain what happened and the fix.**
The graph was compiled without a checkpointer. The pause itself works because pausing needs no storage, but resuming needs the saved checkpoint to restore the run, and there is none. Fix: compile with a checkpointer, `InMemorySaver` for development or a durable saver like `SqliteSaver` for production, and always resume with the same `thread_id`.

**5. On resume, the interrupted node executes again from its first line. Why does this rule exist, and what does it imply for where you place side effects?**
LangGraph resumes by re-entering the node and letting `interrupt()` return the resume value; it does not snapshot the program counter mid-function. So every line before `interrupt()` runs twice. Irreversible side effects (sending email, charging a card, writing to a database) must go after the `interrupt()` call, and anything before it should be cheap preparation or cached in state.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.